# Used Car Price Prediction

## 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

## 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")


In [7]:
df = pd.read_csv(r"data/cardekho_imputated.csv", index_col=[0])

In [8]:
df

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


## Data Cleaning
### Handling Missing values

* Handling Missing values 
* Handling Duplicates
* Check data type
* Understand the dataset

In [9]:
## Check null values

df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [10]:
## Remove Unnecessary columns
df.drop(columns=['car_name','brand'], inplace=True)

In [11]:
df['model'].unique()

<ArrowStringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [12]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


##### Checking count of features

In [13]:
## Numerical Features 

numerical_features = [features for features in df.columns if df[features].dtype != 'str']
print("Numerical Features:", len(numerical_features))

## Categorical Features 
categorical_features = [features for features in df.columns if df[features].dtype == 'str']
print("Categorical Features:",len(categorical_features))

## Discrete Features 
discrete_features = [features for features in numerical_features if len(df[features].unique()) <= 25]
print("Discrete Features:",len(discrete_features))

## Continuous Features 
continuous_features = [features for features in numerical_features if features not in discrete_features]
print("Continuous Features:",len(continuous_features))

Numerical Features: 7
Categorical Features: 4
Discrete Features: 2
Continuous Features: 5


### Independent and Dependent Features

In [14]:
X=df.drop(['selling_price'],axis=1)
y=df['selling_price']

In [15]:
X

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5
...,...,...,...,...,...,...,...,...,...,...
19537,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
19540,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
19541,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
19542,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7


In [16]:
y

0         120000
1         550000
2         215000
3         226000
4         570000
          ...   
19537     250000
19540     925000
19541     425000
19542    1225000
19543    1200000
Name: selling_price, Length: 15411, dtype: int64

## Feature Encoding and Scaling
* One Hot Encoding for Columns which had lesser unique values and not ordinal 
* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [17]:
len(df['model'].unique())

120

In [18]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

#### Encoding

In [19]:
from sklearn.preprocessing import LabelEncoder

lbl_encoder = LabelEncoder()
X['model'] = lbl_encoder.fit_transform(X['model'])
# X['car_name'] = lbl_encoder.fit_transform(X['car_name'])    # Drop if required
# X['brand'] = lbl_encoder.fit_transform(X['brand'])          # Drop if required

In [20]:
X

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5
...,...,...,...,...,...,...,...,...,...,...
19537,117,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
19540,42,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
19541,77,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
19542,114,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7


#### Encoding - Transformers

In [21]:
num_features = X.select_dtypes(exclude='str').columns
cat_features = X.select_dtypes(include='str').columns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# StandardScaler is generally not needed in Random Forest
# ColumnTransformer is used were multiple transformer application is required

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

# preprocessor is the transformation
preprocessor = ColumnTransformer(
    [
        ("Standart Scaler", numeric_transformer, num_features),
        ("One Hot Encoder", oh_transformer, cat_features),
    ], remainder='passthrough'
)

In [22]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Standart Scaler', ...), ('One Hot Encoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transfor

##### Fit Transform on X

In [23]:
X=preprocessor.fit_transform(X)

In [24]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022,1.0,0.0,0.0,0.0,0.0,1.0,1.0
1,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022,1.0,0.0,0.0,0.0,0.0,1.0,1.0
2,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022,1.0,0.0,0.0,0.0,0.0,1.0,1.0
3,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022,1.0,0.0,0.0,0.0,0.0,1.0,1.0
4,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022,0.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022,0.0,0.0,0.0,0.0,0.0,1.0,1.0
15407,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444,0.0,0.0,0.0,0.0,0.0,1.0,1.0
15408,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022,0.0,0.0,1.0,0.0,0.0,0.0,1.0
15409,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444,0.0,0.0,1.0,0.0,0.0,0.0,1.0


### Train Test Split

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

## Model Training and Selection

In [26]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Def Metrics

In [27]:
def evaluate_model(true_value, predicted_value):
    mse = mean_squared_error(true_value, predicted_value)
    mae = mean_absolute_error(true_value, predicted_value)
    rmse = np.sqrt(mean_squared_error(true_value, predicted_value))
    r2_scr = r2_score(true_value, predicted_value)
    return mse, mae, rmse, r2_scr

Model Selection and Training

In [28]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "K Nearest Neighbours": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regression": RandomForestRegressor(),
    "Adaboost Regressor": AdaBoostRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)          # Train Model

    ## Prediction
    y_train_prediction = model.predict(X_train)
    y_test_prediction = model.predict(X_test)

    ## Evaluate Model
    mse_train, mae_train, rmse_train, r2_scr_train = evaluate_model(y_train, y_train_prediction)
    mse_test, mae_test, rmse_test, r2_scr_test = evaluate_model(y_test, y_test_prediction)

    print(list(models.keys())[i])

    print("-"*35)

    print("Model Performance for Training dataset")
    print("- Mean Squared Error: {:.2f}".format(mse_train))
    print("- Mean Absolute Error: {:.2f}".format(mae_train))
    print("- Root Mean Squared Error: {:.2f}".format(rmse_train))
    print("- R2 Score: {:.2f}".format(r2_scr_train))

    print("-"*35)

    print("Model Performance for Test dataset")
    print("- Mean Squared Error: {:.2f}".format(mse_test))
    print("- Mean Absolute Error: {:.2f}".format(mae_test))
    print("- Root Mean Squared Error: {:.2f}".format(rmse_test))
    print("- R2 Score: {:.2f}".format(r2_scr_test))
    print("-"*35)
    print("\n")

Linear Regression
-----------------------------------
Model Performance for Training dataset
- Mean Squared Error: 306756099359.76
- Mean Absolute Error: 268101.61
- Root Mean Squared Error: 553855.67
- R2 Score: 0.62
-----------------------------------
Model Performance for Test dataset
- Mean Squared Error: 252550062888.57
- Mean Absolute Error: 279618.58
- Root Mean Squared Error: 502543.59
- R2 Score: 0.66
-----------------------------------


Ridge
-----------------------------------
Model Performance for Training dataset
- Mean Squared Error: 306756818740.93
- Mean Absolute Error: 268059.80
- Root Mean Squared Error: 553856.32
- R2 Score: 0.62
-----------------------------------
Model Performance for Test dataset
- Mean Squared Error: 252540243247.97
- Mean Absolute Error: 279557.22
- Root Mean Squared Error: 502533.82
- R2 Score: 0.66
-----------------------------------


Lasso
-----------------------------------
Model Performance for Training dataset
- Mean Squared Error: 30675

Observation: Random Forest Performed best and the second best is K Nearest Neighbour so we will hyper parameter tune the two models

Setup Hyper Parameters for KNN and RF

In [29]:
knn_params = {"n_neighbors":[2, 3, 10, 20, 40, 501]}
rf_params = {
    "max_depth": [5, 8, 15, None, 101],
    "max_features": [5, 7, "auto", 8],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, 1000]
}

ada_params={
    "n_estimators":[50,60,70,80],
    "loss":['linear','square','exponential']
}


Setup randomCV

In [30]:
randomcv_models = [
    ("KNN", KNeighborsRegressor(), knn_params),
    ("Random Forest", RandomForestRegressor(), rf_params),
    ("Adaboost Regressor", AdaBoostRegressor(), ada_params)
]

Initialize RandomSearchCV

In [31]:
from sklearn.model_selection import RandomizedSearchCV

model_params={}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,param_distributions=params, n_iter=100, cv=3, verbose=2, n_jobs=1)

    # Model Training
    random.fit(X_train,y_train)
    model_params[name] = random.best_params_

for model_name in model_params:
    print(f"------------------------------- Best Params for {model_name} -------------------------------")
    print(model_params[model_name])

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END ......................................n_neighbors=2; total time=   0.0s
[CV] END ......................................n_neighbors=2; total time=   0.0s
[CV] END ......................................n_neighbors=2; total time=   0.0s
[CV] END ......................................n_neighbors=3; total time=   0.0s
[CV] END ......................................n_neighbors=3; total time=   0.0s
[CV] END ......................................n_neighbors=3; total time=   0.0s
[CV] END .....................................n_neighbors=10; total time=   0.1s
[CV] END .....................................n_neighbors=10; total time=   0.1s
[CV] END .....................................n_neighbors=10; total time=   0.1s
[CV] END .....................................n_neighbors=20; total time=   0.1s
[CV] END .....................................n_neighbors=20; total time=   0.1s
[CV] END .....................................n_n

### Model Re-Trainig 

- Fine tuning with the best parameters after Hyper Parameter Tuning

In [33]:
models_best_params = {
    "K Nearest Neighbours": KNeighborsRegressor(n_neighbors=10),
    "Random Forest Regression": RandomForestRegressor(n_estimators=100, min_samples_split=2, max_features=5, max_depth=101),
    "Adaboost Regressor": AdaBoostRegressor(n_estimators=60, loss='linear')
}

for i in range(len(list(models_best_params))):
    model = list(models_best_params.values())[i]
    model.fit(X_train,y_train)          # Train Model

    ## Prediction
    y_train_prediction = model.predict(X_train)
    y_test_prediction = model.predict(X_test)

    ## Evaluate Model
    mse_train, mae_train, rmse_train, r2_scr_train = evaluate_model(y_train, y_train_prediction)
    mse_test, mae_test, rmse_test, r2_scr_test = evaluate_model(y_test, y_test_prediction)

    print(list(models_best_params.keys())[i])

    print("-"*35)

    print("Model Performance for Training dataset")
    print("- Mean Squared Error: {:.2f}".format(mse_train))
    print("- Mean Absolute Error: {:.2f}".format(mae_train))
    print("- Root Mean Squared Error: {:.2f}".format(rmse_train))
    print("- R2 Score: {:.2f}".format(r2_scr_train))

    print("-"*35)

    print("Model Performance for Test dataset")
    print("- Mean Squared Error: {:.2f}".format(mse_test))
    print("- Mean Absolute Error: {:.2f}".format(mae_test))
    print("- Root Mean Squared Error: {:.2f}".format(rmse_test))
    print("- R2 Score: {:.2f}".format(r2_scr_test))
    print("-"*35)
    print("\n")

K Nearest Neighbours
-----------------------------------
Model Performance for Training dataset
- Mean Squared Error: 132092084047.70
- Mean Absolute Error: 103456.46
- Root Mean Squared Error: 363444.75
- R2 Score: 0.84
-----------------------------------
Model Performance for Test dataset
- Mean Squared Error: 69620968665.46
- Mean Absolute Error: 117432.90
- Root Mean Squared Error: 263857.86
- R2 Score: 0.91
-----------------------------------


Random Forest Regression
-----------------------------------
Model Performance for Training dataset
- Mean Squared Error: 17741935399.03
- Mean Absolute Error: 39431.41
- Root Mean Squared Error: 133198.86
- R2 Score: 0.98
-----------------------------------
Model Performance for Test dataset
- Mean Squared Error: 41987633270.72
- Mean Absolute Error: 97381.46
- Root Mean Squared Error: 204908.84
- R2 Score: 0.94
-----------------------------------


Adaboost Regressor
-----------------------------------
Model Performance for Training datas